# 02 — Data Audit and Cleaning

## Goal

This notebook audits the raw League of Legends match dataset and prepares a clean dataset for downstream analysis.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"../data/raw/league_matches.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24618 entries, 0 to 24617
Columns: 123 entries, match_id to ban_5_champion_id
dtypes: bool(9), float64(10), int64(98), str(6)
memory usage: 22.6 MB


---------------

## 1. Missing Values

In [3]:
df.isna().sum().sort_values(ascending=False).head(20)

team_position        135
ban_3_champion_id     62
ban_1_champion_id     62
ban_2_champion_id     62
ban_4_champion_id     62
ban_5_champion_id     62
champion_level         0
queue_id               0
champion_id            0
game_duration          0
match_id               0
role                   0
lane                   0
win                    0
kills                  0
deaths                 0
assists                0
double_kills           0
triple_kills           0
quadra_kills           0
dtype: int64

No important columns have missing values.

------------

## 2. Match Structure and Duplicates

In [4]:
df.groupby("match_id").size().value_counts().sort_index()

6       1
10    629
16      7
20    882
30     19
Name: count, dtype: int64

In [5]:
#each match should have 10 players; gets rid of duplicates

df = df.drop_duplicates().copy()
df = df.groupby("match_id").filter(lambda x: len(x) == 10)
df.groupby("match_id").size().value_counts()


10    1530
Name: count, dtype: int64

In [6]:
#each team should have 5 players

df.groupby(["match_id", "team_id"]).size().value_counts()

5    3060
Name: count, dtype: int64

------------------

## 3. Position Variables

In [7]:
#comparing repetitive position columns

for col in ["team_position", "individual_position", "role", "lane"]:
    print("\n", col)
    print(df[col].value_counts(dropna=False))


 team_position
team_position
TOP        3060
MIDDLE     3060
JUNGLE     3058
UTILITY    3058
BOTTOM     3054
NaN          10
Name: count, dtype: int64

 individual_position
individual_position
JUNGLE     3070
MIDDLE     3059
TOP        3058
BOTTOM     3052
UTILITY    3051
Invalid      10
Name: count, dtype: int64

 role
role
SOLO       4658
SUPPORT    4141
NONE       2919
CARRY      2560
DUO        1022
Name: count, dtype: int64

 lane
lane
BOTTOM    5045
MIDDLE    3053
JUNGLE    2919
TOP       2763
NONE      1520
Name: count, dtype: int64


`team_position` provides the most directly useful five-position classification for this analysis.

The other fields are less suitable because `role` and `individual_position` use different or less informative classifications, while `lane` groups both ADC and support under `BOTTOM` and includes `NONE`.

For downstream analysis, `team_position` will therefore be used as the primary position variable.

In [8]:
#checking if missing values come from different games

df[df["team_position"].isna()][["match_id", "team_position"]]

,match_id,team_position
2428,EUW1_7442836179,NaN
7051,EUW1_7350821403,NaN
7228,EUW1_7447840096,NaN
8388,EUW1_7425718503,NaN
8934,EUW1_7445251034,NaN
9353,EUW1_7439434166,NaN
9821,EUW1_7348636740,NaN
10374,EUW1_7357348806,NaN
12313,EUW1_7276386738,NaN
22211,EUW1_7340249229,NaN


The missing `team_position` values are limited to individual observations across separate matches rather than being concentrated within a single match. These rows are retained because the remaining match and performance data may still be useful for analyses that do not depend on player position.

----------------

## 4. Data Type and Range Checks

In [9]:
#check that boolean columns have booleans

bool_cols = [
    "win",
    "eligible_for_progression",
    "game_ended_in_surrender",
    "game_ended_in_early_surrender",
    "team_early_surrendered",
    "first_blood_kill",
    "first_blood_assist",
    "first_tower_kill",
    "first_tower_assist",
]

df[bool_cols].dtypes

win                              bool
eligible_for_progression         bool
game_ended_in_surrender          bool
game_ended_in_early_surrender    bool
team_early_surrendered           bool
first_blood_kill                 bool
first_blood_assist               bool
first_tower_kill                 bool
first_tower_assist               bool
dtype: object

In [10]:
#check that nonnegative columns are >= 0

nonnegative_cols = [
    "kills", "deaths", "assists",
    "gold_earned", "gold_spent",
    "total_damage_dealt",
    "total_damage_dealt_to_champions",
    "total_damage_taken",
    "total_minions_killed",
    "neutral_minions_killed",
    "vision_score",
    "wards_placed",
    "wards_killed",
    "dragon_kills",
    "baron_kills",
    "turret_kills",
]

for col in nonnegative_cols:
    if (df[col] < 0).any():
        print(col, "has negatives")

-----------------------

## 5. Redundant Features and Sanity Check


In [11]:
(df["game_duration"] - df["time_played"]).value_counts().head()

0    15300
Name: count, dtype: int64

Game duration = time_played so we only keep one of those columns.

In [12]:
#KDA is calculated correctly; deaths equal to zero are temporarily replaced with missing values to avoid division-by-zero errors 

calculated_kda = (
    (df["kills"] + df["assists"])
    / df["deaths"].replace(0, np.nan)
)

pd.DataFrame({
    "provided": df["kda"],
    "calculated": calculated_kda
}).head(20)

,provided,calculated
0,0.600000,0.600000
1,1.500000,1.500000
2,5.666667,5.666667
3,1.400000,1.400000
4,2.333333,2.333333
5,13.000000,NaN
6,4.750000,4.750000
7,1.800000,1.800000
8,1.333333,1.333333
9,2.000000,2.000000


In [13]:
team_win_consistency = (
    df.groupby(["match_id", "team_id"])["win"]
      .nunique()
      .value_counts()
      .sort_index()
)

team_win_consistency

win
1    3060
Name: count, dtype: int64

In [14]:
team_results = (
    df.groupby(["match_id", "team_id"])["win"]
      .first()
      .groupby("match_id")
      .agg(["sum", "count"])
)

team_results.value_counts()

sum  count
1    2        1530
Name: count, dtype: int64

----------------

## 6. Other

In [15]:
#keep only Ranked Solo/Duo games

df = df[df["queue_id"] == 420]

In [16]:
#remove extremely short games (you can remake until 180 seconds into the match if somebody is AFK)

df = df[df["game_duration"] > 180]

---------------

## 7. Final Dataset

In [17]:
#remove redundant and out-of-scope columns

columns_to_drop = [
    #constant after Ranked Solo/Duo filtering
    "queue_id",

    #redundant
    "champion_skin_id",
    "individual_position",
    "role",
    "lane",
    "time_played",

    #rune configuration
    "rune_primary_style",
    "rune_secondary_style",
    "rune_stat_defense",
    "rune_stat_flex",
    "rune_stat_offense",

    #champion bans
    "ban_1_champion_id",
    "ban_2_champion_id",
    "ban_3_champion_id",
    "ban_4_champion_id",
    "ban_5_champion_id",
]

df = df.drop(columns=columns_to_drop)

In [18]:
df.columns.tolist()

['match_id',
 'game_duration',
 'champion_id',
 'champion_name',
 'champion_level',
 'champion_experience',
 'team_id',
 'team_position',
 'win',
 'kills',
 'deaths',
 'assists',
 'double_kills',
 'triple_kills',
 'quadra_kills',
 'penta_kills',
 'unreal_kills',
 'largest_killing_spree',
 'largest_multi_kill',
 'killing_sprees',
 'total_damage_dealt',
 'total_damage_dealt_to_champions',
 'total_damage_taken',
 'total_damage_shielded_on_teammates',
 'physical_damage_dealt',
 'physical_damage_dealt_to_champions',
 'physical_damage_taken',
 'magic_damage_dealt',
 'magic_damage_dealt_to_champions',
 'magic_damage_taken',
 'true_damage_dealt',
 'true_damage_dealt_to_champions',
 'true_damage_taken',
 'damage_self_mitigated',
 'gold_earned',
 'gold_spent',
 'items_purchased',
 'consumables_purchased',
 'total_minions_killed',
 'neutral_minions_killed',
 'total_ally_jungle_minions_killed',
 'total_enemy_jungle_minions_killed',
 'vision_score',
 'wards_placed',
 'wards_killed',
 'vision_wards_

--------------

## Final Feature Cleanup

After restricting the dataset to Ranked Solo/Duo matches, several redundant or out-of-scope variables are removed.

- `queue_id` is no longer needed because all retained observations belong to the same queue.
- `team_position` is retained as the primary position variable.
- `game_duration` is retained instead of the redundant `time_played` field.
- Rune configuration and champion ban variables are excluded because they fall outside the scope of the analysis.

The remaining columns represent the match, champion, position, outcome, and gameplay-performance features used in downstream analysis.

In [19]:
df.to_csv("../data/processed/matches_clean.csv", index=False)